---
# HACKATHON DEFAULT MINER MODELS
---
###-REGULARIZED LINEAR REGRESSION
### -CART
### -LSTM
---
- Models are easily adjustable to see how parameters change prediction accuracy
- Validator sets test & train and converts to dt time
- Miner receives data and runs basic models


In [ ]:
import pandas as pd
import numpy as np

from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

---
# ADJUSTABLE PARAMETERS:
---

In [ ]:
RANDOM_SEED = 183

# Feature Engineering
USE_TIME_FEATURES = True
USE_CYCLICAL_TIME = True
USE_STATION_AGG_FEATURES = True
USE_TEMP_DEW_GAP = True
USE_WIND_VECTOR_FEATURES = True

# Short-horizon load forecasting features
USE_LOAD_LAG_FEATURES = True
USE_ROLLING_LOAD_FEATURES = False

# 5-minute steps back from the prediction timestamp
LOAD_LAG_STEPS = [1, 2, 3, 6, 12, 24]
ROLLING_LOAD_WINDOWS = [3, 6, 12, 24]

VALIDATION_SPLIT = 0.2

# MLR
LINEAR_MODEL_TYPE = "ridge"  # "ridge", "ols", or "lasso"
ALPHA = 1.0
FIT_INTERCEPT = True
MAX_ITER = 5000

# CART
CART_MAX_DEPTH = 6
CART_MIN_SAMPLES_SPLIT = 10
CART_MIN_SAMPLES_LEAF = 5
CART_RANDOM_STATE = RANDOM_SEED

# LSTM
N_STEPS = 12
LSTM_UNITS = 32
# NN PARAMETERS
DROPOUT_RATE = 0.20
LEARNING_RATE = 0.001
EPOCHS = 20
BATCH_SIZE = 32


In [ ]:
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

---
# DATA LOAD + PREP
---

In [ ]:
# Mount Google Drive so Colab can access the hackathon datasets
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Data prep

In [ ]:
data_path = "/content/drive/MyDrive/hi-resData/"
test_path = "/content/drive/MyDrive/hi-resData/Single-row-scrapes/"

train = pd.read_csv(data_path + "TRAIN-updated-train-data_3-25-26.csv")
test = pd.read_csv(test_path + "TEST-03-25-2026-5min-single.csv")

print("Training shape:", train.shape)
print("Test shape:", test.shape)



if 'load_change_1h' in train.columns:
    train = train.drop(columns=['load_change_1h'])

train.head()

Training shape: (105637, 64)
Test shape: (1, 56)


,dt,Total Load,Native Load,Asset Related Load,Total Load With Estimated Solar,Native Load With Estimated Solar,4B8-drct,4B8-dwpf,4B8-relh,4B8-sped,...,OXC-relh,OXC-sped,OXC-tmpf,SNC-drct,SNC-dwpf,SNC-relh,SNC-sped,SNC-tmpf,hour,month
0,2025-03-23 00:00:00,10740.085,10726.385,13.7,10740.085,10726.385,335.0,20.575,49.6225,11.2125,...,100.0,5.644824,41.653065,325.0,26.60,57.2375,13.5125,40.55,0,3
1,2025-03-23 00:05:00,10709.437,10696.637,12.8,10709.437,10696.637,330.0,19.950,49.4250,10.9250,...,100.0,5.642513,41.647638,320.0,26.60,58.2550,13.2250,40.10,0,3
2,2025-03-23 00:10:00,10675.417,10662.617,12.8,10675.417,10662.617,325.0,19.325,49.2275,10.6375,...,100.0,5.640201,41.642211,315.0,26.60,59.2725,12.9375,39.65,0,3
3,2025-03-23 00:15:00,10663.309,10650.109,13.2,10663.309,10650.109,320.0,18.700,49.0300,10.3500,...,100.0,5.637889,41.636784,310.0,26.60,60.2900,12.6500,39.20,0,3
4,2025-03-23 00:20:00,10629.912,10616.512,13.4,10629.912,10616.512,322.5,18.275,48.6250,10.6375,...,100.0,5.635578,41.631357,315.0,26.15,60.2275,12.6500,38.75,0,3


In [ ]:
test.head()

,dt,4B8-drct,4B8-dwpf,4B8-relh,4B8-sped,4B8-tmpf,BDL-drct,BDL-dwpf,BDL-relh,BDL-sped,...,OXC-drct,OXC-dwpf,OXC-relh,OXC-sped,OXC-tmpf,SNC-drct,SNC-dwpf,SNC-relh,SNC-sped,SNC-tmpf
0,2026-03-25 12:35:00,225.0,20.416667,31.666667,4.166667,49.166667,243.75,20.0,30.666667,6.0,...,225.0,26.0,44.666667,5.583333,46.166667,225.0,25.416667,39.416667,2.583333,49.0


### Datetime conversions

In [ ]:
train["dt"] = pd.to_datetime(train["dt"])
test["dt"] = pd.to_datetime(test["dt"])

---
# FEATURE ENGINEERING:
---

This feature engineering is controlled by the parameter adjustment code at the top of this file. To remove these features, you can either manually edit the code blocks below or simply change `TRUE` to `FALSE` for any of the parameter groups.

In [ ]:
def add_engineered_features(df):
    df = df.copy()

    # Basic time features
    if USE_TIME_FEATURES:
        df["hour"] = df["dt"].dt.hour
        df["minute"] = df["dt"].dt.minute
        df["dayofweek"] = df["dt"].dt.dayofweek
        df["month"] = df["dt"].dt.month

    # Cyclical time features, better than raw hour/minute for many ML models
    if USE_CYCLICAL_TIME:
        df["hour_sin"] = np.sin(2 * np.pi * df["dt"].dt.hour / 24)
        df["hour_cos"] = np.cos(2 * np.pi * df["dt"].dt.hour / 24)

        minute_of_day = df["dt"].dt.hour * 60 + df["dt"].dt.minute
        df["minute_of_day_sin"] = np.sin(2 * np.pi * minute_of_day / 1440)
        df["minute_of_day_cos"] = np.cos(2 * np.pi * minute_of_day / 1440)

    tmpf_cols = [c for c in df.columns if c.endswith("-tmpf")]
    dwpf_cols = [c for c in df.columns if c.endswith("-dwpf")]
    relh_cols = [c for c in df.columns if c.endswith("-relh")]
    sped_cols = [c for c in df.columns if c.endswith("-sped")]
    drct_cols = [c for c in df.columns if c.endswith("-drct")]

    # Aggregate station weather
    if USE_STATION_AGG_FEATURES:
        if len(tmpf_cols) > 0:
            df["tmpf_mean"] = df[tmpf_cols].mean(axis=1)
            df["tmpf_std"] = df[tmpf_cols].std(axis=1)
            df["tmpf_min"] = df[tmpf_cols].min(axis=1)
            df["tmpf_max"] = df[tmpf_cols].max(axis=1)

        if len(relh_cols) > 0:
            df["relh_mean"] = df[relh_cols].mean(axis=1)
            df["relh_std"] = df[relh_cols].std(axis=1)

        if len(sped_cols) > 0:
            df["sped_mean"] = df[sped_cols].mean(axis=1)
            df["sped_std"] = df[sped_cols].std(axis=1)
            df["sped_max"] = df[sped_cols].max(axis=1)

    # Temperature - dewpoint gap
    if USE_TEMP_DEW_GAP and len(tmpf_cols) > 0 and len(dwpf_cols) > 0:
        station_prefixes = sorted(set(c.split("-")[0] for c in tmpf_cols))

        temp_dew_gap_cols = []
        for station in station_prefixes:
            tmp_col = f"{station}-tmpf"
            dwp_col = f"{station}-dwpf"
            if tmp_col in df.columns and dwp_col in df.columns:
                new_col = f"{station}_temp_dew_gap"
                df[new_col] = df[tmp_col] - df[dwp_col]
                temp_dew_gap_cols.append(new_col)

        if len(temp_dew_gap_cols) > 0:
            df["temp_dew_gap_mean"] = df[temp_dew_gap_cols].mean(axis=1)
            df["temp_dew_gap_std"] = df[temp_dew_gap_cols].std(axis=1)

    # Wind direction as vectors
    if USE_WIND_VECTOR_FEATURES and len(drct_cols) > 0:
        wind_x_cols = []
        wind_y_cols = []

        station_prefixes = sorted(set(c.split("-")[0] for c in drct_cols))
        for station in station_prefixes:
            drct_col = f"{station}-drct"
            sped_col = f"{station}-sped"

            if drct_col in df.columns and sped_col in df.columns:
                radians = np.deg2rad(df[drct_col])
                x_col = f"{station}_wind_x"
                y_col = f"{station}_wind_y"

                df[x_col] = df[sped_col] * np.cos(radians)
                df[y_col] = df[sped_col] * np.sin(radians)

                wind_x_cols.append(x_col)
                wind_y_cols.append(y_col)

        if len(wind_x_cols) > 0:
            df["wind_x_mean"] = df[wind_x_cols].mean(axis=1)
            df["wind_y_mean"] = df[wind_y_cols].mean(axis=1)

    # Autoregressive load features: short-horizon signal
    # LOAD_LAG_FEATURES: previous target values at n*5 minute-intervals
    # For example, if LOAD_LAG_STEPS = [1, 3, 5, 10] then the model is using the grid values at n = [5, 15, 25, 50] minutes ago
    if "Total Load" in df.columns:
        if USE_LOAD_LAG_FEATURES:
            for lag in LOAD_LAG_STEPS:
                df[f"load_lag_{lag}"] = df["Total Load"].shift(lag)

    #ROLLING_LOAD_FEATURES
        if USE_ROLLING_LOAD_FEATURES:
            for window in ROLLING_LOAD_WINDOWS:
                shifted_load = df["Total Load"].shift(1)
                df[f"load_roll_mean_{window}"] = shifted_load.rolling(window=window).mean()
                df[f"load_roll_std_{window}"] = shifted_load.rolling(window=window).std()
                df[f"load_roll_min_{window}"] = shifted_load.rolling(window=window).min()
                df[f"load_roll_max_{window}"] = shifted_load.rolling(window=window).max()

            df["load_delta_1"] = df["Total Load"].shift(1) - df["Total Load"].shift(2)
            df["load_delta_3"] = df["Total Load"].shift(1) - df["Total Load"].shift(4)
            df["load_delta_12"] = df["Total Load"].shift(1) - df["Total Load"].shift(13)

    return df


def add_test_load_features_from_history(test_df, history_df):
    test_df = test_df.copy()

    if len(test_df) != 1:
        raise ValueError("This helper expects the single-row future test file.")

    load_hist = history_df["Total Load"].reset_index(drop=True)

    if USE_LOAD_LAG_FEATURES:
        for lag in LOAD_LAG_STEPS:
            if len(load_hist) < lag:
                raise ValueError(f"Not enough history to compute load_lag_{lag}.")
            test_df[f"load_lag_{lag}"] = load_hist.iloc[-lag]

    if USE_ROLLING_LOAD_FEATURES:
        for window in ROLLING_LOAD_WINDOWS:
            if len(load_hist) < window:
                raise ValueError(f"Not enough history to compute rolling window {window}.")
            hist_window = load_hist.iloc[-window:]
            test_df[f"load_roll_mean_{window}"] = hist_window.mean()
            test_df[f"load_roll_std_{window}"] = hist_window.std()
            test_df[f"load_roll_min_{window}"] = hist_window.min()
            test_df[f"load_roll_max_{window}"] = hist_window.max()

        if len(load_hist) < 13:
            raise ValueError("Need at least 13 rows of history for load_delta features.")
        test_df["load_delta_1"] = load_hist.iloc[-1] - load_hist.iloc[-2]
        test_df["load_delta_3"] = load_hist.iloc[-1] - load_hist.iloc[-4]
        test_df["load_delta_12"] = load_hist.iloc[-1] - load_hist.iloc[-13]

    return test_df


In [ ]:
train = add_engineered_features(train)
test = add_engineered_features(test)

# The single future-row test file does not contain current/future Total Load,
# so build its autoregressive load features from the tail of the observed training history.
if USE_LOAD_LAG_FEATURES or USE_ROLLING_LOAD_FEATURES:
    test = add_test_load_features_from_history(test, train)

print("Training shape after feature engineering:", train.shape)
print("Test shape after feature engineering:", test.shape)


Training shape after feature engineering: (105637, 121)
Test shape after feature engineering: (1, 116)


In [ ]:
exclude_cols = [
    "dt",
    "Total Load",
    "Native Load",
    "Asset Related Load",
    "Total Load With Estimated Solar",
    "Native Load With Estimated Solar"
]

features = [col for col in train.columns if col not in exclude_cols]

# Drop the warm-up rows created by lag / rolling features so every model trains on aligned data
required_train_cols = features + ["Total Load"]
train_model = train.dropna(subset=required_train_cols).reset_index(drop=True)

print("Number of features:", len(features))
print(features[:20])
print("Training rows kept after lag/rolling warm-up drop:", len(train_model), "out of", len(train))


Number of features: 115
['4B8-drct', '4B8-dwpf', '4B8-relh', '4B8-sped', '4B8-tmpf', 'BDL-drct', 'BDL-dwpf', 'BDL-relh', 'BDL-sped', 'BDL-tmpf', 'BDR-drct', 'BDR-dwpf', 'BDR-relh', 'BDR-sped', 'BDR-tmpf', 'DXR-drct', 'DXR-dwpf', 'DXR-relh', 'DXR-sped', 'DXR-tmpf']
Training rows kept after lag/rolling warm-up drop: 105613 out of 105637


In [ ]:
X_train = train_model[features].copy()
y_train = train_model["Total Load"].copy()
X_test = test[features].copy()

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)


X_train shape: (105613, 115)
y_train shape: (105613,)
X_test shape: (1, 115)


In [ ]:
split_index = int(len(X_train) * (1 - VALIDATION_SPLIT))

X_train_raw = X_train.iloc[:split_index].copy()
X_val_raw = X_train.iloc[split_index:].copy()

y_train_final = y_train.iloc[:split_index].reset_index(drop=True)
y_val_final = y_train.iloc[split_index:].reset_index(drop=True)

scaler = StandardScaler()
X_train_final = scaler.fit_transform(X_train_raw)
X_val_final = scaler.transform(X_val_raw)
X_test_scaled = scaler.transform(X_test)

print("X_train_final:", X_train_final.shape)
print("X_val_final:", X_val_final.shape)
print("y_train_final:", y_train_final.shape)
print("y_val_final:", y_val_final.shape)
print("X_test_scaled:", X_test_scaled.shape)


X_train_final: (84490, 115)
X_val_final: (21123, 115)
y_train_final: (84490,)
y_val_final: (21123,)
X_test_scaled: (1, 115)


---
# DEFAULT MODEL #1 - LINEAR REG
---


In [ ]:
if LINEAR_MODEL_TYPE == "ridge":
    linear_model = Ridge(alpha=ALPHA, fit_intercept=FIT_INTERCEPT)

elif LINEAR_MODEL_TYPE == "lasso":
    linear_model = Lasso(alpha=ALPHA, fit_intercept=FIT_INTERCEPT, max_iter=MAX_ITER)

elif LINEAR_MODEL_TYPE == "ols":
    linear_model = LinearRegression(fit_intercept=FIT_INTERCEPT)

else:
    raise ValueError("LINEAR_MODEL_TYPE must be one of: 'ridge', 'lasso', 'ols'")

print("Using model:", linear_model)

Using model: Ridge()


In [ ]:
linear_model.fit(X_train_final, y_train_final)

print("Model training complete.")

Model training complete.


### LR Metrics

In [ ]:
val_predictions = linear_model.predict(X_val_final)

print("Number of validation predictions:", len(val_predictions))
print("First 5 validation predictions:", val_predictions[:5])

Number of validation predictions: 21123
First 5 validation predictions: [12502.9174518  12386.14119644 12348.81131987 12318.93135377
 12200.54461364]


In [ ]:
val_rmse = np.sqrt(mean_squared_error(y_val_final, val_predictions))
val_mae = mean_absolute_error(y_val_final, val_predictions)
val_r2 = r2_score(y_val_final, val_predictions)

print("Validation RMSE:", round(val_rmse, 3))
print("Validation MAE:", round(val_mae, 3))
print("Validation R^2:", round(val_r2, 4))

Validation RMSE: 62.119
Validation MAE: 41.071
Validation R^2: 0.9992


In [ ]:
test_prediction = linear_model.predict(X_test_scaled)

print("Number of predictions:", len(test_prediction))
print("Predicted next 5-min Total Load:", test_prediction[0])


Number of predictions: 1
Predicted next 5-min Total Load: 15030.076336601736


In [ ]:
submission = pd.DataFrame({
    "dt": test["dt"],
    "predicted_total_load": test_prediction
})

submission

,dt,predicted_total_load
0,2026-03-25 12:35:00,15030.076337


---
# DEFAULT MODEL 2- CART
---

In [ ]:
cart_model = DecisionTreeRegressor(
    max_depth=CART_MAX_DEPTH,
    min_samples_split=CART_MIN_SAMPLES_SPLIT,
    min_samples_leaf=CART_MIN_SAMPLES_LEAF,
    random_state=CART_RANDOM_STATE
)

print("Using model:", cart_model)

Using model: DecisionTreeRegressor(max_depth=6, min_samples_leaf=5, min_samples_split=10,
                      random_state=183)


In [ ]:
cart_model.fit(X_train_final, y_train_final)

print("Model training complete")

Model training complete


In [ ]:
val_predictions = cart_model.predict(X_val_final)

print("Number of validation predictions:", len(val_predictions))
print("First 5 validation predictions:", val_predictions[:5])

Number of validation predictions: 21123
First 5 validation predictions: [12558.72693468 12412.73689607 12412.73689607 12412.73689607
 12258.86670226]


In [ ]:
val_rmse = np.sqrt(mean_squared_error(y_val_final, val_predictions))
val_mae = mean_absolute_error(y_val_final, val_predictions)
val_r2 = r2_score(y_val_final, val_predictions)

print("Validation RMSE:", round(val_rmse, 3))
print("Validation MAE:", round(val_mae, 3))
print("Validation R^2:", round(val_r2, 4))

Validation RMSE: 102.053
Validation MAE: 80.502
Validation R^2: 0.9978


In [ ]:
test_predictions = cart_model.predict(X_test_scaled)

print("Number of predictions:", len(test_predictions))
print("Predicted next 5-min Total Load:", test_predictions[0])


Number of predictions: 1
Predicted next 5-min Total Load: 15028.110506261193


---
# DEFAULT MODEL #3 - BASIC LSTM

The LSTM now receives lagged / rolling load features inside each timestep, so it is no longer sequence-learning from weather alone.

In [ ]:
def make_sequences(X, y, sequence_length, prediction_horizon=1):
    X_seq = []
    y_seq = []

    max_i = len(X) - sequence_length - prediction_horizon + 1

    for i in range(max_i):
        X_seq.append(X[i:i + sequence_length])
        y_seq.append(y[i + sequence_length + prediction_horizon - 1])

    return np.array(X_seq), np.array(y_seq)

In [ ]:
PREDICTION_HORIZON = 1 # Estimating one 5 minute "step" ahead, do not change

X_seq, y_seq = make_sequences(
    X_train_scaled,
    y_train.values,
    sequence_length=N_STEPS,
    prediction_horizon=PREDICTION_HORIZON
)

print("X_seq shape:", X_seq.shape)
print("y_seq shape:", y_seq.shape)
print("Sequence feature count:", X_seq.shape[2])


X_seq shape: (105601, 12, 115)
y_seq shape: (105601,)
Sequence feature count: 115


In [ ]:
split_index = int(len(X_seq) * (1 - VALIDATION_SPLIT))

X_train_seq = X_seq[:split_index]
X_val_seq = X_seq[split_index:]

y_train_seq = y_seq[:split_index]
y_val_seq = y_seq[split_index:]

print("X_train_seq:", X_train_seq.shape)
print("X_val_seq:", X_val_seq.shape)
print("y_train_seq:", y_train_seq.shape)
print("y_val_seq:", y_val_seq.shape)

X_train_seq: (84480, 12, 115)
X_val_seq: (21121, 12, 115)
y_train_seq: (84480,)
y_val_seq: (21121,)


---
### BUILD & TRAIN THE MODEL:

In [ ]:
lstm_model = Sequential([
    LSTM(LSTM_UNITS, input_shape=(N_STEPS, X_train_seq.shape[2])),
    Dropout(DROPOUT_RATE),
    Dense(16, activation="relu"),
    Dense(1)
])

optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)

lstm_model.compile(
    optimizer=optimizer,
    loss="mse",
    metrics=["mae"]
)

lstm_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 32)             │        18,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,489 (76.13 KB)

 Trainable params: 19,489 (76.13 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = lstm_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/20
2640/2640 ━━━━━━━━━━━━━━━━━━━━ 27s 9ms/step - loss: 131987704.0000 - mae: 11036.6826 - val_loss: 109092816.0000 - val_mae: 10153.2910
Epoch 2/20
2640/2640 ━━━━━━━━━━━━━━━━━━━━ 19s 7ms/step - loss: 29530292.0000 - mae: 4418.2979 - val_loss: 37687620.0000 - val_mae: 4789.5576
Epoch 3/20
2640/2640 ━━━━━━━━━━━━━━━━━━━━ 18s 7ms/step - loss: 5318725.5000 - mae: 1613.1119 - val_loss: 12463832.0000 - val_mae: 1981.1635
Epoch 4/20
2640/2640 ━━━━━━━━━━━━━━━━━━━━ 19s 7ms/step - loss: 2220593.5000 - mae: 1098.0537 - val_loss: 7849396.5000 - val_mae: 1387.5708
Epoch 5/20
2640/2640 ━━━━━━━━━━━━━━━━━━━━ 21s 8ms/step - loss: 1739837.8750 - mae: 1016.2311 - val_loss: 2006212.1250 - val_mae: 607.3585
Epoch 6/20
2640/2640 ━━━━━━━━━━━━━━━━━━━━ 19s 7ms/step - loss: 1512493.1250 - mae: 956.8956 - val_loss: 2844393.2500 - val_mae: 658.2104
Epoch 7/20
2640/2640 ━━━━━━━━━━━━━━━━━━━━ 19s 7ms/step - loss: 1428166.1250 - mae: 923.9878 - val_loss: 5093908.0000 - val_mae: 881.2276
Epoch 8/20
2640/2640 ━━

---
### EVALUATE:

In [ ]:
val_predictions = lstm_model.predict(X_val_seq).flatten()

val_rmse = np.sqrt(mean_squared_error(y_val_seq, val_predictions))
val_mae = mean_absolute_error(y_val_seq, val_predictions)
val_r2 = r2_score(y_val_seq, val_predictions)

print("Validation RMSE:", round(val_rmse, 3))
print("Validation MAE:", round(val_mae, 3))
print("Validation R^2:", round(val_r2, 3))

661/661 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
Validation RMSE: 286.801
Validation MAE: 211.698
Validation R^2: 0.982
